# Assignment 1 — Classic IR End-to-End Flow

**Course:** Information Retrieval  
**Name:** Yakira Siboni  
**Date:** May 2025

---

Full classic IR pipeline on a real corpus fetched live from Wikipedia:

1. **Document collection** — 220 Wikipedia article summaries across 10 real-world topics
2. **Preprocessing** — tokenization, case folding, stop-word removal, stemming
3. **Indexing** — TF-IDF inverted index via `sklearn`
4. **Retrieval** — vector space model, cosine similarity
5. **Query expansion** — automatic Rocchio algorithm (top-3 pseudo-relevant docs)

## 0. Install & Import

In [1]:
!pip install nltk scikit-learn pandas numpy -q

import re
import json
import urllib.request
import numpy as np
import pandas as pd
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

stemmer = PorterStemmer()
print('All imports OK')

All imports OK


---
## 1. Document Collection

I use the **Wikipedia REST API** (`https://en.wikipedia.org/api/rest_v1/page/summary/{title}`)  
to fetch the introductory summary of 220 real Wikipedia articles across 10 topics:

| Topic | Example articles |
|-------|------------------|
| History | World War II, French Revolution, Roman Empire, Cold War … |
| Science | Evolution, DNA, Black hole, Quantum mechanics … |
| Technology | Artificial intelligence, Blockchain, Robotics … |
| Geography | Amazon rainforest, Himalayas, Great Barrier Reef … |
| Politics | Democracy, United Nations, Human rights … |
| Health | Cancer, COVID-19, Alzheimer's disease, Vaccination … |
| Culture | Jazz, Cinema, Hip-hop, Classical music … |
| Economics | Inflation, Stock market, Cryptocurrency … |
| Environment | Climate change, Deforestation, Biodiversity … |
| Philosophy | Ethics, Existentialism, Consciousness … |

Each API call returns a clean plain-text extract — no HTML, no markup.
The `extract` field is the opening paragraph(s) of the article, typically 2–6 sentences.

In [2]:
# 10 topics × 22 article titles each = 220 documents
TITLES = {
    "History": [
        "World_War_II",
        "French_Revolution",
        "Roman_Empire",
        "Cold_War",
        "American_Civil_War",
        "Ottoman_Empire",
        "Black_Death",
        "Mongol_Empire",
        "Renaissance",
        "Industrial_Revolution",
        "Ancient_Egypt",
        "Byzantine_Empire",
        "Age_of_Enlightenment",
        "Crusades",
        "Atlantic_slave_trade",
        "Russian_Revolution",
        "British_Empire",
        "Alexander_the_Great",
        "Napoleon",
        "Thirty_Years%27_War",
        "Spanish_Inquisition",
        "Vietnam_War"
    ],
    "Science": [
        "Evolution",
        "General_relativity",
        "Quantum_mechanics",
        "DNA",
        "Photosynthesis",
        "Big_Bang",
        "Black_hole",
        "Periodic_table",
        "Thermodynamics",
        "Electromagnetism",
        "Cell_(biology)",
        "CRISPR",
        "Stem_cell",
        "Antibiotic",
        "Vaccination",
        "Neuroscience",
        "Dark_matter",
        "String_theory",
        "Higgs_boson",
        "Nuclear_fission",
        "Plate_tectonics",
        "Human_genome"
    ],
    "Technology": [
        "Artificial_intelligence",
        "Internet",
        "Smartphone",
        "Blockchain",
        "Cloud_computing",
        "Machine_learning",
        "Robotics",
        "Semiconductor",
        "Virtual_reality",
        "Cybersecurity",
        "5G",
        "Quantum_computing",
        "Autonomous_vehicle",
        "3D_printing",
        "Augmented_reality",
        "Internet_of_things",
        "Electric_vehicle",
        "Renewable_energy",
        "Nuclear_power",
        "Space_tourism",
        "Nanotechnology",
        "Biotechnology"
    ],
    "Geography": [
        "Amazon_rainforest",
        "Sahara",
        "Himalayas",
        "Atlantic_Ocean",
        "Great_Barrier_Reef",
        "Nile",
        "Mount_Everest",
        "Antarctica",
        "Gobi_Desert",
        "Amazon_River",
        "Pacific_Ocean",
        "Alps",
        "Sahel",
        "Mariana_Trench",
        "Greenland",
        "Congo_rainforest",
        "Andes",
        "Great_Wall_of_China",
        "Mississippi_River",
        "Mediterranean_Sea",
        "Dead_Sea",
        "Yellowstone_National_Park"
    ],
    "Politics": [
        "Democracy",
        "United_Nations",
        "European_Union",
        "NATO",
        "Human_rights",
        "Globalization",
        "Socialism",
        "Capitalism",
        "Federalism",
        "International_law",
        "Geopolitics",
        "Political_philosophy",
        "Diplomacy",
        "Imperialism",
        "Nationalism",
        "Liberalism",
        "Conservatism",
        "Anarchism",
        "Communism",
        "Fascism",
        "Welfare_state",
        "Universal_suffrage"
    ],
    "Health": [
        "Cancer",
        "Diabetes",
        "HIV/AIDS",
        "COVID-19",
        "Cardiovascular_disease",
        "Alzheimer%27s_disease",
        "Depression_(mood)",
        "Obesity",
        "Malaria",
        "Tuberculosis",
        "Mental_health",
        "Immune_system",
        "Nutrition",
        "Exercise",
        "Sleep",
        "Antibiotic_resistance",
        "Pandemic",
        "Vaccine",
        "Public_health",
        "Genetics",
        "Microbiome",
        "Stress_(biology)"
    ],
    "Culture": [
        "Jazz",
        "Renaissance_art",
        "Cinema",
        "Literature",
        "Philosophy",
        "Mythology",
        "Religion",
        "Folklore",
        "Hip_hop_music",
        "Classical_music",
        "Opera",
        "Theatre",
        "Photography",
        "Architecture",
        "Fashion",
        "Video_game",
        "Anime",
        "Comics",
        "Podcast",
        "Social_media",
        "Graffiti",
        "Street_art"
    ],
    "Economics": [
        "Capitalism",
        "Inflation",
        "Gross_domestic_product",
        "Stock_market",
        "International_trade",
        "Supply_and_demand",
        "Monetary_policy",
        "Fiscal_policy",
        "Poverty",
        "Inequality",
        "Cryptocurrency",
        "Central_bank",
        "Microeconomics",
        "Macroeconomics",
        "Behavioral_economics",
        "Game_theory",
        "Labor_economics",
        "Globalization",
        "Free_trade",
        "Economic_inequality",
        "Foreign_direct_investment",
        "Austerity"
    ],
    "Environment": [
        "Climate_change",
        "Deforestation",
        "Biodiversity",
        "Ocean_acidification",
        "Air_pollution",
        "Water_pollution",
        "Ozone_layer",
        "Greenhouse_gas",
        "Coral_reef",
        "Endangered_species",
        "Recycling",
        "Carbon_footprint",
        "Sustainable_development",
        "Desertification",
        "Wildfire",
        "Plastic_pollution",
        "Sea_level_rise",
        "Permafrost",
        "Wind_power",
        "Solar_energy",
        "Carbon_capture",
        "Rewilding"
    ],
    "Philosophy": [
        "Ethics",
        "Epistemology",
        "Metaphysics",
        "Logic",
        "Existentialism",
        "Utilitarianism",
        "Stoicism",
        "Plato",
        "Aristotle",
        "Immanuel_Kant",
        "Friedrich_Nietzsche",
        "Free_will",
        "Consciousness",
        "Moral_relativism",
        "Political_philosophy",
        "Philosophy_of_mind",
        "Phenomenology",
        "Pragmatism",
        "Empiricism",
        "Rationalism",
        "Nihilism",
        "Humanism"
    ]
}

def fetch_wikipedia_summary(title):
    """Fetch the plain-text summary of a Wikipedia article via the REST API."""
    url = 'https://en.wikipedia.org/api/rest_v1/page/summary/' + title
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'IR-assignment/1.0 (educational; contact: student@university.edu)'}
    )
    with urllib.request.urlopen(req, timeout=8) as r:
        data = json.loads(r.read())
    return {
        'title':   data.get('title', title),
        'text':    data.get('extract', ''),
        'url':     data.get('content_urls', {}).get('desktop', {}).get('page', ''),
    }

documents = []
failed    = []

for topic, title_list in TITLES.items():
    for title in title_list:
        try:
            doc = fetch_wikipedia_summary(title)
            if len(doc['text']) > 50:
                documents.append({'id': len(documents), 'topic': topic, **doc})
        except Exception as e:
            failed.append((topic, title, str(e)))

print(f'Fetched  : {len(documents)} documents')
print(f'Failed   : {len(failed)}')
if failed:
    for topic, title, err in failed[:5]:
        print(f'  [{topic}] {title} — {err}')

print()
for idx in [0, 22, 88]:
    if idx < len(documents):
        print(f'--- Doc #{idx}: [{documents[idx]["topic"]}] {documents[idx]["title"]} ---')
        print(documents[idx]['text'][:300])
        print()

Fetched  : 218 documents
Failed   : 0

--- Doc #0: [History] World War II ---
World War II, or the Second World War, was a global conflict between two coalitions: the Allies and the Axis powers. Nearly all of the world's countries participated. Tanks and aircraft played major roles, the latter enabling the strategic bombing of cities and delivery of the only nuclear weapons u

--- Doc #22: [Science] Evolution ---
Evolution is the change in the heritable characteristics of biological populations over successive generations. It occurs when evolutionary processes such as genetic drift and natural selection act on genetic variation, resulting in certain characteristics becoming more or less common within a popul

--- Doc #88: [Politics] Democracy ---
Democracy is a form of government in which political power is vested in the people or the population of a state. Under a minimalist definition of democracy, rulers are elected through competitive elections while more expansive or maximalist de

---
## 2. Preprocessing

Four steps applied in sequence:

| Step | What it does |
|------|--------------|
| **Tokenization** | `re.findall(r'[a-zA-Z]+', text)` — extract alphabetic tokens |
| **Case folding** | `.lower()` — normalise capitalisation |
| **Stop-word removal** | drop common function words and tokens shorter than 3 chars |
| **Stemming** | `nltk.stem.PorterStemmer` — reduce words to their root form |

Demonstrated step-by-step on doc #0 below.

In [3]:
STOP_WORDS = set("""
a an the is are was were be been being have has had do does did will would could
should may might shall can need dare ought used to in on at by for of with about
against between into through during before after above below from up down out off
over under again further then once here there when where why how all both each few
more most other some such no nor not only same so than too very just but also this
that these those i me my we our you your he him his herself she her hers it its
they them their what which who whom am as if or and any
""".split())

def tokenize(text):
    """Step 1: extract alphabetic tokens"""
    return re.findall(r'[a-zA-Z]+', text)

def case_fold(tokens):
    """Step 2: lowercase everything"""
    return [t.lower() for t in tokens]

def remove_stopwords(tokens):
    """Step 3: drop stop words and tokens shorter than 3 chars"""
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]

def apply_stemming(tokens):
    """Step 4: Porter stemmer from NLTK"""
    return [stemmer.stem(t) for t in tokens]

def preprocess(text):
    return apply_stemming(remove_stopwords(case_fold(tokenize(text))))

# ---- step-by-step demo on doc #0 ----
sample = documents[0]['text']
s1 = tokenize(sample)
s2 = case_fold(s1)
s3 = remove_stopwords(s2)
s4 = apply_stemming(s3)
removed = sorted(set(s2[:25]) - set(s3))

print(f'=== Preprocessing Demo ===')
print(f'Doc: [{documents[0]["topic"]}] {documents[0]["title"]}')
print(f'\nRaw text (first 200 chars):\n  "{sample[:200]}"\n')
print(f'Step 1 — Tokenization ({len(s1)} tokens):')
print(f'  {s1[:12]}')
print(f'\nStep 2 — Case Folding:')
print(f'  {s2[:12]}')
print(f'\nStep 3 — Stop-word Removal  (removed from first 25: {removed}):')
print(f'  {s3[:12]}')
print(f'\nStep 4 — Stemming:')
print(f'  {s4[:12]}')
print()
print('Before vs. After Stemming (first 10 terms):')
df_stem = pd.DataFrame(list(zip(s3[:10], s4[:10])), columns=['Before', 'After (stemmed)'])
print(df_stem.to_string(index=False))

=== Preprocessing Demo ===
Doc: [History] World War II

Raw text (first 200 chars):
  "World War II, or the Second World War, was a global conflict between two coalitions: the Allies and the Axis powers. Nearly all of the world's countries participated. Tanks and aircraft played major r"

Step 1 — Tokenization (104 tokens):
  ['World', 'War', 'II', 'or', 'the', 'Second', 'World', 'War', 'was', 'a', 'global', 'conflict']

Step 2 — Case Folding:
  ['world', 'war', 'ii', 'or', 'the', 'second', 'world', 'war', 'was', 'a', 'global', 'conflict']

Step 3 — Stop-word Removal  (removed from first 25: ['a', 'all', 'and', 'between', 'ii', 'of', 'or', 'the', 'was']):
  ['world', 'war', 'second', 'world', 'war', 'global', 'conflict', 'two', 'coalitions', 'allies', 'axis', 'powers']

Step 4 — Stemming:
  ['world', 'war', 'second', 'world', 'war', 'global', 'conflict', 'two', 'coalit', 'alli', 'axi', 'power']

Before vs. After Stemming (first 10 terms):
    Before After (stemmed)
     world          

In [4]:
# preprocess all documents
preprocessed_corpus = []
for doc in documents:
    doc['tokens'] = preprocess(doc['text'])
    preprocessed_corpus.append(' '.join(doc['tokens']))

avg = np.mean([len(d['tokens']) for d in documents])
print(f'Total documents      : {len(documents)}')
print(f'Avg tokens / doc     : {avg:.1f}')
print(f'Raw unique tokens    (doc #0): {len(set(s1))}')
print(f'Stemmed unique tokens (doc #0): {len(set(s4))}')
print(f'\nDoc #0 final tokens: {documents[0]["tokens"][:15]}')

Total documents      : 218
Avg tokens / doc     : 48.3
Raw unique tokens    (doc #0): 73
Stemmed unique tokens (doc #0): 51

Doc #0 final tokens: ['world', 'war', 'second', 'world', 'war', 'global', 'conflict', 'two', 'coalit', 'alli', 'axi', 'power', 'nearli', 'world', 'countri']


---
## 3. Indexing — TF-IDF

`TfidfVectorizer` with `sublinear_tf=True` applies $1 + \log(\text{tf})$ — the log-normalised TF formula from lectures.  
The output is a sparse document-term matrix that represents the inverted index.

$$\text{TF-IDF}(t, d) = (1 + \log \text{TF}(t,d)) \times \log\frac{N}{\text{DF}(t)}$$

In [5]:
vectorizer = TfidfVectorizer(
    token_pattern=r'\S+',  # already preprocessed — split on whitespace
    sublinear_tf=True,      # 1 + log(tf)
)

tfidf_matrix = vectorizer.fit_transform(preprocessed_corpus)
vocab        = vectorizer.get_feature_names_out()
term_to_idx  = {t: i for i, t in enumerate(vocab)}

sparsity = 100 * (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))
print(f'TF-IDF matrix   : {tfidf_matrix.shape}  (docs × vocab)')
print(f'Non-zero entries: {tfidf_matrix.nnz:,}   sparsity: {sparsity:.1f}%')

# show sample inverted index entries
print('\n--- Sample Index Entries (term → top-3 docs by TF-IDF weight) ---')
sample_terms = ['war', 'gene', 'market', 'climat', 'brain']
rows = []
for term in sample_terms:
    if term in term_to_idx:
        col  = tfidf_matrix[:, term_to_idx[term]].toarray().flatten()
        top3 = col.argsort()[::-1][:3]
        top3_str = ', '.join(
            f"{documents[i]['title']} [{documents[i]['topic'][:6]}] ({col[i]:.3f})"
            for i in top3 if col[i] > 0
        )
        rows.append({'Stemmed term': term, 'DF': int((col > 0).sum()), 'Top-3 postings': top3_str})
print(pd.DataFrame(rows).to_string(index=False))

TF-IDF matrix   : (218, 2796)  (docs × vocab)
Non-zero entries: 8,173   sparsity: 98.7%

--- Sample Index Entries (term → top-3 docs by TF-IDF weight) ---
Stemmed term  DF                                                                                       Top-3 postings
         war  12      Thirty Years' War [Histor] (0.349), Vietnam War [Histor] (0.286), World War II [Histor] (0.268)
        gene   2                                 Genetics [Health] (0.232), Antimicrobial resistance [Health] (0.137)
      market  11 Labour economics [Econom] (0.319), Stock market [Econom] (0.256), Supply and demand [Econom] (0.240)
      climat   6             Climate change [Enviro] (0.243), Sahel [Geogra] (0.229), Sea level rise [Enviro] (0.130)
       brain   1                                                                               Sleep [Health] (0.191)


---
## 4. Retrieval — Vector Space Model

For each query:
1. Apply the same preprocessing pipeline
2. Transform into TF-IDF space with `vectorizer.transform()`
3. Score all documents with `cosine_similarity`
4. Return top-K ranked by score

I test 4 queries — each targeting a different topic — to verify the system retrieves the right articles.

In [6]:
def retrieve(query_text, top_k=5):
    """Preprocess query → TF-IDF vector → cosine similarity → top-K"""
    q_tokens = preprocess(query_text)
    print(f'Preprocessed query: {q_tokens}')
    q_vec  = vectorizer.transform([' '.join(q_tokens)])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top    = scores.argsort()[::-1][:top_k]
    return [(i, documents[i]['topic'], documents[i]['title'], float(scores[i]))
            for i in top if scores[i] > 0]

def show_results(query, results):
    print(f'\nQuery: "{query}"')
    print(f'{"Rank":<5} {"Score":<8} {"Topic":<16} Article')
    print('-' * 65)
    for rank, (doc_id, topic, title, score) in enumerate(results, 1):
        print(f'{rank:<5} {score:.4f}   {topic:<16} {title}')
    print()

demo_queries = [
    'world war soldiers empire battle',
    'gene DNA cell evolution organism',
    'inflation market economy trade',
    'climate carbon emissions warming ocean',
]

saved = {}
for q in demo_queries:
    print('=' * 55)
    results = retrieve(q, top_k=5)
    show_results(q, results)
    saved[q] = results

Preprocessed query: ['world', 'war', 'soldier', 'empir', 'battl']

Query: "world war soldiers empire battle"
Rank  Score    Topic            Article
-----------------------------------------------------------------
1     0.3239   History          Thirty Years' War
2     0.1623   History          World War II
3     0.1311   History          Byzantine Empire
4     0.1240   History          Alexander the Great
5     0.1176   History          Cold War

Preprocessed query: ['gene', 'dna', 'cell', 'evolut', 'organ']

Query: "gene DNA cell evolution organism"
Rank  Score    Topic            Article
-----------------------------------------------------------------
1     0.2296   Health           Genetics
2     0.2184   Science          Stem cell
3     0.1913   Science          Cell (biology)
4     0.1663   Science          Human genome
5     0.1482   Science          CRISPR

Preprocessed query: ['inflat', 'market', 'economi', 'trade']

Query: "inflation market economy trade"
Rank  Score    Top

### 4.1 Candidate Set Size

Only docs that share at least one stemmed query term get a non-zero cosine score.

In [7]:
q = demo_queries[0]
q_vec  = vectorizer.transform([' '.join(preprocess(q))])
scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
cands  = (scores > 0).sum()
print(f'Query      : "{q}"')
print(f'Corpus size: {len(documents)} docs')
print(f'Candidates (score > 0)    : {cands}')
print(f'Skipped (no shared terms) : {len(documents) - cands}')

Query      : "world war soldiers empire battle"
Corpus size: 218 docs
Candidates (score > 0)    : 46
Skipped (no shared terms) : 172


---
## 5. Query Expansion — Rocchio Algorithm

Automatic pseudo-relevance feedback — no user needed.

1. Run the original query, take the **top-3** results as pseudo-relevant set $R$
2. Expand the query vector toward the centroid of $R$:

$$\vec{q}_{\text{new}} = \alpha \cdot \vec{q}_{\text{old}} + \frac{\beta}{|R|} \sum_{d \in R} \vec{d}$$

3. Re-rank all documents with the expanded vector

Parameters: $\alpha = 1.0$, $\beta = 0.75$, $|R| = 3$.

In [8]:
def rocchio_retrieve(query_text, top_k=5, pseudo_k=3, alpha=1.0, beta=0.75):
    """
    Rocchio query expansion:
      1. Initial retrieval → pseudo-relevant set (top pseudo_k docs)
      2. Rocchio formula  → expand query vector toward their centroid
      3. Re-rank          → score all docs with expanded vector
    """
    q_vec = vectorizer.transform([' '.join(preprocess(query_text))]).toarray()

    # step 1 — initial retrieval
    init_scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    rel_ids     = init_scores.argsort()[::-1][:pseudo_k]

    # step 2 — Rocchio expansion
    rel_mat = tfidf_matrix[rel_ids].toarray()
    q_exp   = alpha * q_vec + (beta / pseudo_k) * rel_mat.sum(axis=0, keepdims=True)

    # step 3 — re-rank with expanded query
    new_scores = cosine_similarity(q_exp, tfidf_matrix).flatten()
    top        = new_scores.argsort()[::-1][:top_k]
    results    = [(i, documents[i]['topic'], documents[i]['title'], float(new_scores[i]))
                  for i in top if new_scores[i] > 0]
    return results, rel_ids, q_vec, q_exp

feature_names = vectorizer.get_feature_names_out()

for q in demo_queries:
    print('=' * 68)
    print(f'QUERY: "{q}"')

    orig                                 = saved[q]
    exp_results, rel_ids, q_orig, q_exp  = rocchio_retrieve(q, top_k=5)

    # original query terms
    top_t = q_orig.flatten().argsort()[::-1][:5]
    print(f'\nOriginal query terms (top-5 by weight):')
    print(' ', [(feature_names[i], round(float(q_orig.flatten()[i]), 4))
                for i in top_t if q_orig.flatten()[i] > 0])

    # pseudo-relevant docs used
    print(f'\nPseudo-relevant set (top-3 initial hits fed into Rocchio):')
    for rank, idx in enumerate(rel_ids, 1):
        print(f'  [{rank}] [{documents[idx]["topic"]}] {documents[idx]["title"]} — {documents[idx]["text"][:80]}...')

    # terms boosted by expansion
    diff    = q_exp.flatten() - q_orig.flatten()
    new_top = diff.argsort()[::-1][:5]
    print(f'\nTerms boosted most by Rocchio:')
    print(' ', [(feature_names[i], round(float(diff[i]), 4))
                for i in new_top if diff[i] > 0])

    # before / after ranking comparison
    print(f'\n{"Rank":<5} {"BEFORE expansion":<30} {"AFTER expansion"}')
    print('-' * 68)
    for rank in range(5):
        b_id, b_topic, b_title, b_sc = orig[rank]        if rank < len(orig)        else (None,'-','-',0)
        a_id, a_topic, a_title, a_sc = exp_results[rank] if rank < len(exp_results) else (None,'-','-',0)
        tag = '' if b_id == a_id else '<-- changed'
        print(f'{rank+1:<5} {b_title:<30} {a_title}  {tag}')
    print()

QUERY: "world war soldiers empire battle"

Original query terms (top-5 by weight):
  [('soldier', 0.5746), ('battl', 0.5337), ('war', 0.3858), ('empir', 0.3783), ('world', 0.3049)]

Pseudo-relevant set (top-3 initial hits fed into Rocchio):
  [1] [History] Thirty Years' War — The Thirty Years' War, fought primarily in Central Europe between 1618 and 1648,...
  [2] [History] World War II — World War II, or the Second World War, was a global conflict between two coaliti...
  [3] [History] Byzantine Empire — The Byzantine Empire, also known as the Eastern Roman Empire, was the continuati...

Terms boosted most by Rocchio:
  [('war', 0.1543), ('conflict', 0.1088), ('roman', 0.0978), ('empir', 0.0866), ('byzantin', 0.0756)]

Rank  BEFORE expansion               AFTER expansion
--------------------------------------------------------------------
1     Thirty Years' War              Thirty Years' War  
2     World War II                   World War II  
3     Byzantine Empire               By

---
## Summary

| Component | Implementation |
|-----------|---------------|
| **Documents** | 220 Wikipedia article summaries fetched live via `https://en.wikipedia.org/api/rest_v1/page/summary/{title}` |
| **Topics** | History, Science, Technology, Geography, Politics, Health, Culture, Economics, Environment, Philosophy |
| **Tokenization** | `re.findall(r'[a-zA-Z]+', text)` |
| **Case Folding** | `.lower()` |
| **Stop-word removal** | Custom 80-word list |
| **Stemming** | `nltk.stem.PorterStemmer` |
| **Indexing** | `sklearn.feature_extraction.text.TfidfVectorizer` (`sublinear_tf=True`) |
| **Retrieval** | `sklearn.metrics.pairwise.cosine_similarity` |
| **Query Expansion** | Rocchio ($\\alpha=1.0$, $\\beta=0.75$), top-3 pseudo-relevant docs, fully automatic |

The corpus is fetched fresh every run — no static data anywhere in this notebook.